# MKWii RL Evaluation Monitor
Set `RUN` below to the folder name inside `runs_backup/`, then run all cells in order.

In [1]:
import os

# --- Configure which model to evaluate ---
RUN_NUMBER          = 7  # folder name inside runs_backup/
RUN                 = f"run{RUN_NUMBER}"
NUM_EPISODES        = 20
# -----------------------------------------

PROJECT_ROOT = os.path.normpath(os.path.join(os.getcwd(), "..", ".."))
MODEL_PATH   = os.path.join(PROJECT_ROOT, "runs_backup", RUN, "agent_model.pth")
TB_LOGDIR    = os.path.join(PROJECT_ROOT, "runs_backup", RUN, "runs")

print(f"Model : {MODEL_PATH}")
print(f"Exists: {os.path.exists(MODEL_PATH)}")
print(f"TB dir: {TB_LOGDIR}")

Model : c:\Users\Nicolas\LocalDocuments\HSLU\DSPRO2\runs_backup\run7\agent_model.pth
Exists: True
TB dir: c:\Users\Nicolas\LocalDocuments\HSLU\DSPRO2\runs_backup\run7\runs


In [ ]:
import subprocess, sys

# Kill any TensorBoard already running on 6006
subprocess.run(["taskkill", "/f", "/fi", "IMAGENAME eq tensorboard*"], capture_output=True)
subprocess.run(
    ["powershell", "-Command",
     "Get-NetTCPConnection -LocalPort 6006 -ErrorAction SilentlyContinue | "
     "ForEach-Object { Stop-Process -Id $_.OwningProcess -Force -ErrorAction SilentlyContinue }"],
    capture_output=True
)

tb_proc = subprocess.Popen(
    [sys.executable, "-m", "tensorboard.main", "--logdir", TB_LOGDIR, "--port", "6006"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
import time; time.sleep(2)
print(f"TensorBoard running at http://localhost:6006")
print(f"Showing training logs for {RUN}")
print(f"Logdir: {TB_LOGDIR}")

In [3]:
import re
import matplotlib.pyplot as plt
import matplotlib
from IPython.display import display
import ipywidgets as widgets

matplotlib.rcParams['figure.figsize'] = (12, 5)

START_EVAL = os.path.join(PROJECT_ROOT, "scripts", "eval", "start_eval.py")

p1_rewards, p2_rewards = [], []
p1_episodes, p2_episodes = [], []

PATTERN = re.compile(r'\[Evaluate\] P(\d) ep (\d+)/\d+  finished=(\w+)  reward=([\-\d\.]+)')

plot_output = widgets.Output()
display(plot_output)

def update_plot():
    with plot_output:
        plot_output.clear_output(wait=True)
        fig, ax = plt.subplots()
        if p1_rewards:
            ax.plot(p1_episodes, p1_rewards, 'o-', color='#00E5FF', label='P1', linewidth=1.5, markersize=4)
        if p2_rewards:
            ax.plot(p2_episodes, p2_rewards, 'o-', color='#FF6B6B', label='P2', linewidth=1.5, markersize=4)
        ax.axhline(y=0, color='white', linestyle='--', alpha=0.3)
        ax.set_title(f'Eval Episode Reward — {RUN}', color='white')
        ax.set_xlabel('Episode', color='white')
        ax.set_ylabel('Total Reward', color='white')
        ax.legend()
        ax.set_facecolor('#1a1a2e')
        fig.patch.set_facecolor('#0f0f23')
        ax.tick_params(colors='white')
        for spine in ['bottom', 'left']:
            ax.spines[spine].set_color('white')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        plt.tight_layout()
        plt.show()

def parse_line(line):
    line = line.strip()
    if not line:
        return
    print(line, flush=True)
    for m in PATTERN.findall(line):
        player, ep, finished, reward = int(m[0]), int(m[1]), m[2], float(m[3])
        if player == 1:
            p1_rewards.append(reward)
            p1_episodes.append(ep)
        else:
            p2_rewards.append(reward)
            p2_episodes.append(ep)
        update_plot()

print(f"Launching evaluation: {RUN} — {NUM_EPISODES} episodes")
proc = subprocess.Popen(
    [sys.executable, "-u", START_EVAL,
     "--model", MODEL_PATH,
     "--episodes", str(NUM_EPISODES)],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    for line in proc.stdout:
        parse_line(line)
except KeyboardInterrupt:
    proc.terminate()
    print("Evaluation stopped.")

proc.wait()
print("Evaluation process exited.")

Output()

Launching evaluation: run7 — 20 episodes
[StartEval] Model: c:\Users\Nicolas\LocalDocuments\HSLU\DSPRO2\runs_backup\run7\agent_model.pth
[StartEval] Launching evaluate.py...
[NeuralAgent] Loaded model from c:\Users\Nicolas\LocalDocuments\HSLU\DSPRO2\runs_backup\run7\agent_model.pth (steps_done reset to 0 for full eps warmup)
[NeuralAgent] Failed to load replay buffer: Ran out of input
[NeuralAgent] This run will be backed up as run10
[Evaluate] Starting up...
[Evaluate] Ports bound, ready file written.
[Evaluate] Waiting for Dolphin...
[StartEval] evaluate.py ready.
[StartEval] Launching Dolphin...
[StartEval] Running evaluation...
[Evaluate] P1 connected.
[Evaluate] P2 connected.
[Evaluate] Dolphin window not ready, retry 1/20...
[Evaluate] Dolphin window not ready, retry 1/20...
[DolphinCapture] Player 1 ready.
[DolphinCapture] Player 2 ready.
[Evaluate] P1 ep 1/20  finished=True  reward=31.88


[Evaluate] P1 ep 2/20  finished=True  reward=0.00


[Evaluate] P2 ep 1/20  finished=False  reward=20.88


[Evaluate] P1 ep 3/20  finished=False  reward=14.15


...

[Evaluate] P1 ep 19/20  finished=False  reward=-11.87


[Evaluate] P2 ep 19/20  finished=False  reward=5.74


[Evaluate] P1 ep 20/20  finished=False  reward=-17.87


[Evaluate] P1 COMPLETE â€” finish_rate=30.0%  avg_reward=4.76  best=31.88
[Evaluate] P2 ep 20/20  finished=False  reward=-14.26


[Evaluate] P2 COMPLETE â€” finish_rate=30.0%  avg_reward=2.80  best=35.30
EVALUATION COMPLETE  (20 episodes per player)
P1: finish_rate=30.0%  avg_reward=4.76  best=31.88
P2: finish_rate=30.0%  avg_reward=2.80  best=35.30
[StartEval] Done.
Evaluation process exited.
